# Transcript → time ranges (starter)

The simplest first step: **import a transcript and parse it into `time range` + `text`.**
Everything else (confidence, inferred type, dependencies) builds on top of this table later.

The corpus transcripts carry `speaker` / `text` but **no timestamps**, so we estimate
each turn's duration from its word count at a speaking rate (`WORDS_PER_MINUTE`) and
accumulate. If a transcript *does* carry real `start_time` / `end_time` on a turn
(e.g. from a real STT engine later), those are used as-is instead.

## 1 · Config — pick a transcript

In [1]:
from pathlib import Path
import json
import pandas as pd

# Find the repo root whether we're launched from notebooks/ or the repo root.
_here = Path.cwd()
REPO = next(p for p in (_here, *_here.parents) if (p / "research" / "corpus").exists())

TRANSCRIPT = REPO / "research" / "corpus" / "hospital-bed-mgmt" / "transcript.json"
WORDS_PER_MINUTE = 150   # average speaking pace; tune to taste
GAP_SECONDS = 0.5        # small pause inserted between turns

print("transcript:", TRANSCRIPT.relative_to(REPO))

transcript: research/corpus/hospital-bed-mgmt/transcript.json


## 2 · Parse into a time-ranged table

One row per turn: `start` / `end` (seconds), a human `time_range`, the speaker, and the text.

In [2]:
def mmss(seconds: float) -> str:
    """Seconds -> M:SS (or H:MM:SS past an hour)."""
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"


def load_turns(path: Path) -> pd.DataFrame:
    data = json.loads(Path(path).read_text())
    turns = data["turns"] if isinstance(data, dict) else data

    rows, clock = [], 0.0
    for i, turn in enumerate(turns):
        text = (turn.get("text") or "").strip()
        n_words = len(text.split())

        # Prefer real timestamps if the turn has them; else estimate from words.
        if turn.get("start_time") is not None and turn.get("end_time") is not None:
            start, end = float(turn["start_time"]), float(turn["end_time"])
        else:
            start = clock
            end = start + n_words / WORDS_PER_MINUTE * 60.0
            clock = end + GAP_SECONDS

        rows.append({
            "turn": i,
            "start": round(start, 1),
            "end": round(end, 1),
            "time_range": f"{mmss(start)}–{mmss(end)}",
            "speaker": turn.get("speaker_name") or turn.get("speaker") or "unknown",
            "words": n_words,
            "text": text,
        })
    return pd.DataFrame(rows)


df = load_turns(TRANSCRIPT)
print(f"{len(df)} turns · ~{mmss(df['end'].max())} total")
df[["time_range", "speaker", "text"]]

8 turns · ~5:37 total


,time_range,speaker,text
0,0:00–0:39,Dr. Naomi Fields,"Good morning, everyone. Today I want to walk y..."
1,0:40–1:23,Dr. Naomi Fields,Let me describe how bed assignment actually wo...
2,1:24–1:58,Dr. Naomi Fields,The exceptions are where it really falls apart...
3,1:58–2:36,Dr. Naomi Fields,So here's what I need this system to actually ...
4,2:37–3:12,Dr. Naomi Fields,"On the technical side, everything has to integ..."
5,3:13–4:00,Dr. Naomi Fields,"Now, constraints, and there are some real ones..."
6,4:01–4:42,Dr. Naomi Fields,A couple of things I genuinely don't have answ...
7,4:43–5:37,Dr. Naomi Fields,"Let me just play back what I've described, bec..."


## 3 · Slice by time

Once it's a DataFrame, filtering to any window is one line.

In [3]:
def between(df, start_s, end_s):
    """Turns that overlap the [start_s, end_s) window."""
    return df[(df["end"] > start_s) & (df["start"] < end_s)]

# example: everything spoken in the first 60 seconds
between(df, 0, 60)[["time_range", "speaker", "text"]]

,time_range,speaker,text
0,0:00–0:39,Dr. Naomi Fields,"Good morning, everyone. Today I want to walk y..."
1,0:40–1:23,Dr. Naomi Fields,Let me describe how bed assignment actually wo...


## From here

This `df` (time range + text) is the base layer. Next steps build columns onto it:

- `confidence` — how sure the model is about a turn's derived meaning
- `artifact_type` — what it was inferred to be (objective, requirement, constraint, …)
- `dependencies` — reinforcement vs. conflict with other turns

The richer, pipeline-driven version of all that already lives in
`inference_explorer.ipynb` — this starter is the minimal on-ramp to it.